# Tugas Kelompok 2: Membangun Model CNN dan Arsitektur Jaringan

**Tujuan:**
Merancang dan melatih Convolutional Neural Network (CNN) yang efektif untuk klasifikasi gambar pada dataset CIFAR-10 (10 kelas objek).

**Cakupan:**
membangun, meringkas, dan meng-compile arsitektur dan melatih model secara end-to-end pada CIFAR-10, serta melakukan evaluasi pada test set yang tidak digunakan saat training, serta menganalisis performa per kelas menggunakan confusion matrix dan contoh kesalahan klasifikasi (misclassification).

**Filosofi Desain:**
1. **Blok Double Conv2D** untuk meningkatkan kedalaman representasi tanpa downsampling spasial yang agresif.
2. **Batch Normalization** setelah setiap Conv2D untuk menstabilkan training.
3. **Progressive Dropout + L2 weight decay** sebagai strategi regularisasi berlapis untuk mencegah overfitting pada input kecil (32x32).


### Bagian 0: Setup dan Reproduktibilitas

Kami mendefinisikan satu konstanta `SEED` dan meneruskannya ke `train_test_split` untuk memastikan bahwa pembagian train/validation identik di setiap kali run.

**Cakupan reproduktibilitas (mohon dibaca dengan teliti):**
- Reproducible: sampel mana yang masuk ke `X_train` vs `X_val`.
- *Tidak* reproducible: inisialisasi weight, dropout mask, randomness pada augmentasi. Oleh karena itu, akurasi test akhir akan berfluktuasi sekitar +/-1% di setiap run, tetapi pembagian datanya tetap.

Kami sengaja tidak mengaktifkan `tf.config.experimental.enable_op_determinism()` karena perlambatan training sebesar 1.5x-2x tidak sepadan untuk tugas akademik.


In [ ]:
SEED = 42

import tensorflow as tf
from tensorflow.keras import datasets, layers, models, regularizers, callbacks
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print(f"Versi TensorFlow:    {tf.__version__}")
print(f"GPU tersedia:        {len(tf.config.list_physical_devices('GPU')) > 0}")
print(f"Random seed:         {SEED}")


### Bagian 1: Memuat Dataset CIFAR-10

CIFAR-10 berisi 60.000 gambar RGB berukuran 32x32 dari 10 kelas, dengan 50.000 sampel training dan 10.000 sampel test (pembagian test telah ditetapkan oleh penulis dataset dan kami tidak akan mengubahnya sepanjang notebook ini).


In [ ]:
(train_images, train_labels), (test_images, test_labels) = datasets.cifar10.load_data()

class_names = ['pesawat', 'mobil', 'burung', 'kucing', 'rusa',
               'anjing', 'katak', 'kuda', 'kapal', 'truk']

print("Data berhasil dimuat!")
print(f"Jumlah sampel training: {len(train_images)}")
print(f"Jumlah sampel test:     {len(test_images)}")
print(f"Ukuran gambar:          {train_images.shape[1:]}")
print(f"Jumlah kelas:           {len(class_names)}")


### Bagian 2: Preprocessing dan Stratified Train/Validation Split

Tiga langkah preprocessing:

1. **melakukan normalisasi piksel (0-1).** Mengubah skala nilai piksel 8-bit dari [0, 255] menjadi [0, 1] demi stabilitas numerik dan agar selaras dengan asumsi implisit dari weight initializer He/Glorot.
2. **Stratified train/validation split.** Memotong validation set dari 50.000 sampel training menggunakan `sklearn.model_selection.train_test_split` dengan `stratify=y_train` untuk memastikan ke-10 kelas terwakili secara seimbang di kedua split. Kami *tidak* menyentuh 10.000 sampel test asli.
3. **Encoding label one-hot.** Mengubah label integer (0-9) menjadi vektor one-hot 10 dimensi, sebagaimana dipersyaratkan oleh `categorical_crossentropy`.

**notes:** stratifikasi harus menggunakan label *integer*, bukan one-hot. Oleh karena itu kami melakukan split terlebih dahulu, baru kemudian one-hot encoding.


In [ ]:
# 1. Normalisasi nilai piksel ke [0, 1]
X_train_full = train_images.astype('float32') / 255.0
X_test       = test_images.astype('float32') / 255.0
y_train_full = train_labels.flatten()   # Pertahankan sebagai integer untuk stratifikasi
y_test_int   = test_labels.flatten()

# 2. Stratified train/validation split (10% disisihkan untuk validasi)
X_train, X_val, y_train_int, y_val_int = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.1,
    random_state=SEED,
    stratify=y_train_full
)

# 3. One-hot encoding label untuk categorical_crossentropy
y_train = to_categorical(y_train_int, num_classes=10)
y_val   = to_categorical(y_val_int,   num_classes=10)
y_test  = to_categorical(y_test_int,  num_classes=10)

print("Split dan preprocessing selesai.")
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_val shape:   {X_val.shape}, y_val shape:   {y_val.shape}")
print(f"X_test shape:  {X_test.shape}, y_test shape:  {y_test.shape}")
print(f"\nDistribusi kelas pada validation set (target ~500 per kelas):")
unique, counts = np.unique(y_val_int, return_counts=True)
for cls, count in zip(unique, counts):
    print(f"  {class_names[cls]:12s}: {count}")


### Bagian 3: Pipeline Data Augmentation

Augmentation diterapkan melalui pipeline `tf.data`, bukan sebagai layer di dalam model. kerana:

1. **Menjaga `model.summary()` tetap bersih.** Tugas ini berfokus pada arsitektur, sehingga augmentation tidak boleh muncul sebagai layer arsitektur.
2. **Augmentation hanya diterapkan pada training set.** Data validation dan test harus dievaluasi pada gambar yang tidak diaugmentasi, jika tidak metrik yang dihasilkan akan menjadi target yang berfluktuasi alih-alih ukuran generalisasi yang sesungguhnya.

Kami menggunakan augmentation ringan (rotasi dan zoom kecil, horizontal flip) karena augmentation yang agresif pada gambar 32x32 akan merusak fitur diskriminatif. Pelajaran ini dibawa langsung dari tugas sebelumnya (cat-vs-dog), di mana augmentation berlebihan menyebabkan akurasi training tertahan di sekitar 50%.


In [ ]:
# Pipeline augmentation (hanya diterapkan pada data training)
augmenter = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
], name='augmenter')

batch_size = 64

# Membangun tf.data datasets
train_ds = (tf.data.Dataset.from_tensor_slices((X_train, y_train))
            .shuffle(buffer_size=10000, seed=SEED)
            .batch(batch_size)
            .map(lambda x, y: (augmenter(x, training=True), y),
                 num_parallel_calls=tf.data.AUTOTUNE)
            .prefetch(tf.data.AUTOTUNE))

val_ds = (tf.data.Dataset.from_tensor_slices((X_val, y_val))
          .batch(batch_size)
          .prefetch(tf.data.AUTOTUNE))

test_ds = (tf.data.Dataset.from_tensor_slices((X_test, y_test))
           .batch(batch_size)
           .prefetch(tf.data.AUTOTUNE))

print("Datasets siap:")
print(f"  train_ds: {len(X_train)} sampel, dengan augmentation")
print(f"  val_ds:   {len(X_val)} sampel, tanpa augmentation")
print(f"  test_ds:  {len(X_test)} sampel, tanpa augmentation")


### Bagian 4: Membangun Arsitektur CNN

CNN Sequential dengan tiga blok konvolusional yang diikuti oleh classifier fully-connected. Setiap blok menggandakan jumlah filter (32 -> 64 -> 128) sambil membagi dua dimensi spasial melalui MaxPooling, mengikuti pola desain VGG standar.

**Struktur blok:**
`Conv2D -> BatchNorm -> Conv2D -> BatchNorm -> MaxPooling2D -> Dropout`

**Stack regularisasi:**
- L2 weight decay (`1e-4`) pada setiap layer Conv2D dan Dense
- Batch Normalization setelah setiap Conv2D dan setelah layer dense classifier
- Progressive Dropout: 0.2 -> 0.3 -> 0.4 -> 0.5 (layer yang lebih dalam menerima regularisasi lebih kuat)


In [ ]:
weight_decay = 1e-4

model = models.Sequential([
    layers.Input(shape=(32, 32, 3)),

    # Blok 1: 32 filter, 32x32 -> 16x16
    layers.Conv2D(32, (3, 3), padding='same', activation='relu',
                  kernel_regularizer=regularizers.l2(weight_decay)),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3, 3), padding='same', activation='relu',
                  kernel_regularizer=regularizers.l2(weight_decay)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.2),

    # Blok 2: 64 filter, 16x16 -> 8x8
    layers.Conv2D(64, (3, 3), padding='same', activation='relu',
                  kernel_regularizer=regularizers.l2(weight_decay)),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), padding='same', activation='relu',
                  kernel_regularizer=regularizers.l2(weight_decay)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.3),

    # Blok 3: 128 filter, 8x8 -> 4x4
    layers.Conv2D(128, (3, 3), padding='same', activation='relu',
                  kernel_regularizer=regularizers.l2(weight_decay)),
    layers.BatchNormalization(),
    layers.Conv2D(128, (3, 3), padding='same', activation='relu',
                  kernel_regularizer=regularizers.l2(weight_decay)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.4),

    # Classifier head
    layers.Flatten(),
    layers.Dense(128, activation='relu',
                 kernel_regularizer=regularizers.l2(weight_decay)),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')   # 10 output untuk klasifikasi objek
])


### Bagian 5: Menampilkan Arsitektur Model

In [ ]:
model.summary()

### Bagian 6: Meng-compile Model

Compile dengan optimizer **Adam** (learning rate adaptif, default yang robust untuk vision) dan loss **categorical_crossentropy** (pilihan natural untuk klasifikasi multi-kelas dengan label one-hot dan output softmax).


In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Model berhasil di-compile.")
print(f"Optimizer: {model.optimizer.__class__.__name__}")
print(f"Loss:      {model.loss}")
print(f"Metrics:   {[m.name for m in model.metrics]}")


### Bagian 7: Melatih Model

Dua callback mengatur loop training:

- **EarlyStopping** (`patience=10`, `restore_best_weights=True`): jika validation loss tidak membaik selama 10 epoch berturut-turut, training berhenti dan weight model di-rollback ke epoch dengan validasi terbaik. Patience diset cukup besar agar `ReduceLROnPlateau` punya waktu untuk bertindak terlebih dahulu.
- **ReduceLROnPlateau** (`patience=3`, `factor=0.5`): jika validation loss mendatar selama 3 epoch, learning rate dipotong setengah (dengan batas bawah `1e-6`). Ini memungkinkan model melakukan fine-tuning dengan langkah lebih kecil setelah loss landscape menjadi datar.

Kami menetapkan `epochs=50` sebagai batas atas; EarlyStopping biasanya akan menghentikan training antara epoch 25-35.


In [ ]:
callback_list = [
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        patience=3,
        factor=0.5,
        min_lr=1e-6,
        verbose=1
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=callback_list,
    verbose=1
)

print(f"\nTraining selesai setelah {len(history.history['loss'])} epoch.")
print(f"Validation loss terbaik:     {min(history.history['val_loss']):.4f}")
print(f"Validation accuracy terbaik: {max(history.history['val_accuracy']):.4f}")


### Bagian 8: Riwayat Training

Plot akurasi dan loss untuk training dan validation set sepanjang epoch. Bentuk kurva ini adalah diagnostik utama untuk overfitting (kurva val menjauh ke atas dari kurva train pada loss, ke bawah pada accuracy) atau underfitting (kedua kurva mendatar di akurasi rendah).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Kurva accuracy
axes[0].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Akurasi Model')
axes[0].legend(loc='lower right')
axes[0].grid(alpha=0.3)

# Kurva loss
axes[1].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Loss Model')
axes[1].legend(loc='upper right')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


### Bagian 9: Evaluasi pada Test Set

Validation set memandu training (early stopping, learning rate decay) sehingga *bukan* merupakan estimasi generalisasi yang tidak bias. Test set sebanyak 10.000 sampel, yang tidak disentuh hingga sekarang, adalah ukuran yang tidak bias.

Kami melaporkan dua tingkatan metrik:
1. **Agregat**: test loss dan accuracy keseluruhan.
2. **Per kelas**: precision, recall, dan F1-score untuk masing-masing dari 10 kelas melalui `classification_report`. Ini menunjukkan kelas mana yang ditangani model dengan baik dan kelas mana yang menjadi kesulitan.


In [ ]:
# Metrik agregat
print("Mengevaluasi pada test set...")
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"\nTest Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}\n")

# Metrik per kelas
y_pred_proba = model.predict(test_ds, verbose=0)
y_pred = np.argmax(y_pred_proba, axis=1)
y_true = y_test_int

print("Laporan klasifikasi per kelas:")
print("-" * 65)
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))


### Bagian 10: Confusion Matrix

Confusion matrix 10x10 menunjukkan pasangan kelas mana yang paling sering dirancukan oleh model. CIFAR-10 memiliki pasangan-pasangan sulit yang sudah dikenal:

- **kucing vs. anjing**: tekstur bulu dan pose yang serupa
- **mobil vs. truk**: keduanya kendaraan darat dengan bentuk yang mirip
- **rusa vs. kuda**: keduanya hewan berkaki empat di latar luar ruang
- **burung vs. pesawat**: keduanya muncul dengan latar belakang langit

Diagonal menunjukkan prediksi yang benar; sel di luar diagonal menunjukkan kekeliruan yang sistematis.


In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Jumlah'})
plt.xlabel('Label Prediksi')
plt.ylabel('Label Sebenarnya')
plt.title('Confusion Matrix - Test Set CIFAR-10')
plt.tight_layout()
plt.show()


### Bagian 11: Misclassification dengan Confidence Tertinggi

Daripada memvisualisasikan kesalahan secara acak, kami menampilkan **9 gambar test di mana model paling yakin pada prediksi yang salah** (probabilitas prediksi tertinggi di antara sampel-sampel yang salah klasifikasi).

Kasus-kasus ini lebih bernilai diagnostik dibandingkan kesalahan acak: kasus-kasus ini mengungkap kegagalan *sistematis* pada representasi yang dipelajari model, bukan sekadar noise di perbatasan keputusan. Setiap judul menunjukkan `Sebenarnya -> Prediksi (confidence)`.


In [ ]:
# Identifikasi semua sampel yang salah klasifikasi
misclassified_mask = (y_pred != y_true)
misclassified_indices = np.where(misclassified_mask)[0]

# Untuk setiap sampel yang salah klasifikasi, ambil confidence pada kelas yang diprediksi (salah)
predicted_confidence = y_pred_proba[misclassified_indices, y_pred[misclassified_indices]]

# Urutkan berdasarkan confidence menurun, ambil 9 teratas
top9_local_idx = np.argsort(predicted_confidence)[-9:][::-1]
top9_global_idx = misclassified_indices[top9_local_idx]

# Visualisasi
plt.figure(figsize=(12, 12))
for i, idx in enumerate(top9_global_idx):
    plt.subplot(3, 3, i + 1)
    plt.imshow(X_test[idx])
    true_class = class_names[y_true[idx]]
    pred_class = class_names[y_pred[idx]]
    confidence = y_pred_proba[idx, y_pred[idx]]
    plt.title(f"{true_class} -> {pred_class}\n(confidence: {confidence:.1%})",
              fontsize=11)
    plt.axis('off')

plt.suptitle('9 Misclassification dengan Confidence Tertinggi', fontsize=14, y=1.00)
plt.tight_layout()
plt.show()


### Bagian 12: Laporan Desain Arsitektur

Bagian ini menjelaskan tiga keputusan desain inti yang dipersyaratkan oleh tugas: **jumlah layer**, **ukuran kernel**, dan **penggunaan dropout**.

---

#### 12.1 Pemilihan Jumlah Layer

Arsitektur menggunakan **tiga blok konvolusional** dengan **dua layer Conv2D per blok** (total enam layer konvolusional), ditambah classifier head yang fully-connected. Alasannya:

- **Batasan spasial dari ukuran input.** Gambar CIFAR-10 hanya berukuran 32x32. Setiap layer MaxPooling2D membagi dua dimensi spasial: 32 -> 16 -> 8 -> 4. Menambah blok keempat akan memperkecil feature map menjadi 2x2, menyisakan terlalu sedikit informasi spasial untuk konvolusi yang bermakna. Tiga blok karenanya merupakan batas atas yang berprinsip untuk ukuran input ini.
- **Double Conv2D per blok (pola VGG).** Menumpuk dua konvolusi 3x3 sebelum pooling memberikan receptive field efektif 5x5 dengan parameter lebih sedikit dan satu non-linearitas tambahan dibandingkan dengan satu konvolusi 5x5. Ini adalah penalaran yang sama yang memotivasi desain VGGNet dan cocok untuk gambar berukuran kecil dengan fitur lokal.
- **Pertumbuhan filter progresif (32 -> 64 -> 128).** Menggandakan jumlah filter saat dimensi spasial mengecil menjaga biaya komputasi per layer tetap seimbang sembari memungkinkan layer yang lebih dalam menangkap kombinasi fitur yang lebih abstrak.
- **Classifier head ringan.** Satu layer Dense(128) sudah cukup karena vektor flatten 4x4x128 = 2048 sudah merupakan representasi tingkat tinggi yang kompak. Menumpuk beberapa layer Dense besar cenderung menyebabkan overfitting pada CIFAR-10 tanpa peningkatan akurasi yang sepadan.

#### 12.2 Ukuran Kernel

Semua layer konvolusional menggunakan **kernel 3x3**. Ini adalah pilihan yang disengaja dibanding kernel yang lebih besar (5x5, 7x7) dengan tiga alasan:

- **Efisiensi parameter.** Kernel 3x3 dengan C input/output channel memiliki 9C^2 parameter, dibanding 25C^2 untuk kernel 5x5. Dua layer 3x3 yang ditumpuk (18C^2 parameter) mencapai receptive field 5x5 yang sama dengan parameter lebih sedikit dan satu non-linearitas ReLU tambahan.
- **Fitur halus.** Pada input 32x32, fitur diskriminatif (tepi telinga hewan, siluet sayap) memang kecil. Kernel 3x3 cocok dengan skala ini tanpa merata-ratakan terlalu banyak konteks di sekitarnya.
- **Konvensi empiris.** Kernel 3x3 telah menjadi standar de facto sejak VGGNet (2014) dan tetap dominan dalam arsitektur modern (ResNet, EfficientNet). Ini adalah default yang aman dan telah teruji.

#### 12.3 Penggunaan Dropout

Dropout diterapkan di empat titik dengan **tingkat yang meningkat secara progresif** (0.2, 0.3, 0.4, 0.5):

- **Mengapa tingkat progresif?** Layer konvolusional awal mempelajari fitur tingkat rendah yang generik (tepi, warna, tekstur) yang berguna di semua kelas; dropout yang agresif di sini akan membuang sinyal yang dibutuhkan oleh sisa jaringan. Layer yang lebih dalam mempelajari pola yang lebih khusus per kelas yang juga lebih rentan terhadap memorization, sehingga lebih toleran (dan diuntungkan) dari dropout yang lebih berat.
- **Mengapa 0.5 di layer Dense?** Classifier head memiliki kepadatan parameter tertinggi (koneksi Flatten -> Dense(128) saja memiliki ~262 ribu parameter, hampir setengah dari total parameter model). Di sinilah risiko overfitting tertinggi, sehingga tingkat dropout 0.5 yang diusulkan oleh Srivastava dkk. (2014) sangat tepat.
- **Penempatan dropout: setelah pooling, sebelum blok berikutnya.** Menempatkan dropout setelah MaxPooling2D (alih-alih di antara dua layer Conv2D) menghindari gangguan pada akumulasi receptive field di dalam blok. Dropout berfungsi sebagai gerbang regularisasi antar blok, bukan di dalam blok.
- **Saling melengkapi dengan regularizer lain.** Dropout dipasangkan dengan L2 weight decay (`1e-4`) dan BatchNormalization. Ketiganya melayani tujuan yang berbeda: L2 menghukum weight yang besar, BatchNorm menstabilkan aktivasi, dan Dropout memaksa redundansi pada representasi fitur. Bersama-sama mereka membentuk pertahanan berlapis terhadap overfitting yang tidak dapat dicapai oleh teknik tunggal manapun.


### Bagian 13: Diskusi Hasil Training

> **Catatan:** Bagian ini adalah kerangka diskusi. Setelah training selesai, isi placeholder dalam kurung `[...]` dengan angka aktual Anda, dan pilih bullet pada setiap subseksi yang sesuai dengan pengamatan Anda.

#### 13.1 Perilaku Konvergensi

- **Akurasi test akhir:** `[XX,X%]` (diisi dari Bagian 9).
- **Jumlah epoch yang dilatih:** `[NN]` (training berhenti via EarlyStopping pada epoch `[NN]`).
- **Selisih train vs validation:** `[selisih%]` pada epoch terakhir. Selisih kecil (<5%) menunjukkan stack regularisasi (Dropout + L2 + BatchNorm + augmentation) bekerja sesuai desain; selisih besar (>10%) akan mengindikasikan model masih overfitting meskipun sudah diregularisasi.
- **Efek `ReduceLROnPlateau`:** learning rate diturunkan pada epoch `[NN]`, setelah itu validation loss [terus membaik / mendatar]. Ini mengonfirmasi bahwa langkah yang lebih kecil [bermanfaat / bukan bottleneck] pada fase training akhir.

#### 13.2 Performa Per Kelas

Dari classification report dan confusion matrix:

- **Kelas dengan performa terbaik:** biasanya `mobil`, `kapal`, `truk` (kelas kendaraan memiliki geometri yang kaku dan latar belakang yang khas). Pastikan apakah run Anda sesuai.
- **Kelas dengan performa terburuk:** biasanya `kucing`, `anjing`, `burung`. Kelas hewan memiliki tekstur bulu/bulu yang serupa, pose yang mirip, dan latar belakang alami.
- **Pasangan kekeliruan terkuat di matrix:** `[misal: kucing -> anjing dengan NN error, rusa -> kuda dengan NN error]`. Ini sesuai dengan pola kesulitan CIFAR-10 yang sudah dikenal.

#### 13.3 Wawasan dari Misclassification dengan Confidence Tertinggi

Visualisasi pada Bagian 11 biasanya menunjukkan:

- **Ambiguitas pose:** hewan yang difoto dari sudut tidak biasa menyerupai spesies lain pada siluetnya.
- **Latar belakang mendominasi:** beberapa sampel salah klasifikasi terutama karena petunjuk dari latar belakang (langit -> burung/pesawat, air -> kapal).
- **Petunjuk warna menyesatkan model:** kucing hitam mungkin diklasifikasikan sebagai anjing jika kucing dalam training set rata-rata berwarna lebih cerah.

#### 13.4 Peningkatan untuk Iterasi Selanjutnya

Langkah-langkah konkrit yang kemungkinan dapat menaikkan akurasi melampaui `[XX,X%]` saat ini:

- **Transfer learning** dengan backbone yang sudah dipretraining (MobileNetV2, ResNet50). Memerlukan upscaling input 32x32 menjadi 96x96 atau lebih besar untuk memenuhi persyaratan input model pretrained.
- **Augmentation yang lebih kuat** (Mixup, CutMix, RandAugment). Augmentation ringan saat ini bersifat konservatif; setelah model cukup teregularisasi, teknik-teknik ini dapat menambah generalisasi.
- **Peningkatan arsitektur:** mengganti `Flatten + Dense` dengan `GlobalAveragePooling2D`, yang secara dramatis mengurangi parameter dan cenderung memperbaiki generalisasi pada dataset kecil.
- **Ensemble** dari 3-5 model yang dilatih secara independen (seed berbeda), merata-ratakan prediksi. Biasanya menambah 1-2% test accuracy dengan biaya waktu training.
